# Week 6: 움직임 평가와 모션캡처
## Movement Assessment and Motion Capture

**학습목표:**
- 관절 각도(Joint Angle) 분석
- 운동 범위(ROM: Range of Motion) 측정
- 정상 vs 치료 후 움직임 비교
- 임상적 운동 품질 평가


## ⚙️ 환경 설정

In [ ]:
# ── 환경 설정 ──────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
from datetime import datetime
from pathlib import Path

WEEK = 6
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/2026_lecture/Digital_health_pt')
else:
    BASE = Path('.')            # 로컬 실행 시 현재 폴더 사용

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_OUT = BASE / f'{WEEK}week' / 'output' / f'run_{timestamp}'
RUN_OUT.mkdir(parents=True, exist_ok=True)

print(f'출력 경로: {RUN_OUT}')


## 📦 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── 시각화 전역 설정 ─────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})
sns.set_style('whitegrid')

# ── 공통 상수 ────────────────────────────────────────────────
FS          = 100          # Hz
CYCLE_DUR   = 1.0          # 보행 사이클 길이 (초)
N_CYCLES    = 10           # 총 보행 사이클 수
DURATION    = N_CYCLES * CYCLE_DUR
NORMAL_ROMS = {'Hip': 50, 'Knee': 60, 'Ankle': 35}   # 정상 ROM 참고값(도)
JOINT_COLS  = ['hip_angle', 'knee_angle', 'ankle_angle']
JOINT_NAMES = ['Hip', 'Knee', 'Ankle']
JOINT_COLORS = {'Hip': '#E74C3C', 'Knee': '#27AE60', 'Ankle': '#2980B9'}

# ── 공통 유틸 ────────────────────────────────────────────────
def save_fig(fig, name: str):
    """그림을 출력 폴더에 저장하고 경로를 출력."""
    path = RUN_OUT / name
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'그림 저장: {path}')

def save_csv(df: pd.DataFrame, name: str):
    """DataFrame을 CSV로 저장하고 경로를 출력."""
    path = RUN_OUT / name
    df.to_csv(path, index=False)
    print(f'CSV 저장: {path}')

def make_time_gait(duration=DURATION, fs=FS, cycle_dur=CYCLE_DUR):
    """(시간 배열, 보행 위상 0~100%) 반환."""
    t = np.arange(0, duration, 1 / fs)
    phase = (t % cycle_dur) / cycle_dur * 100   # 0~100
    return t, phase

print('라이브러리 및 설정 완료')


---
## 🟢 초급 (Beginner)
### 1. 단일 관절 각도 분석: 무릎 굴곡

**보행 사이클 단계:**

| 단계 | % | 설명 |
|------|---|------|
| IC | 0 % | Initial Contact (초기 접촉) |
| LR | 12 % | Loading Response (부하 반응) |
| MSt | 50 % | Mid Stance (중간 입각기) |
| TSt | 88 % | Terminal Stance (말기 입각기) |
| PSw | 100 % | Pre Swing (전 유각기) |


### 1-1. 무릎 각도 데이터 생성 및 ROM 계산

In [ ]:
np.random.seed(42)
t, phase = make_time_gait()

# 무릎 각도: 5°(IC) → 최대 20°(60 % 유각기) 정현파 근사
knee_angle = 5 + 15 * np.sin(np.pi * phase / 100) + 1.0 * np.random.randn(len(t))

knee_df = pd.DataFrame({'time': t, 'gait_phase': phase, 'knee_angle': knee_angle})

# ── ROM 계산 ─────────────────────────────────────────────────
knee_max, knee_min = knee_df['knee_angle'].max(), knee_df['knee_angle'].min()
knee_rom  = knee_max - knee_min
normal_rom = NORMAL_ROMS['Knee']

rom_result = pd.DataFrame({
    'Measure':  ['Maximum', 'Minimum', 'ROM', 'Normal ROM', 'ROM % of Normal'],
    'Value':    [knee_max, knee_min, knee_rom, normal_rom, knee_rom / normal_rom * 100],
    'Unit':     ['°', '°', '°', '°', '%'],
})

print('무릎 ROM 분석:')
print(rom_result.to_string(index=False))


### 1-2. 관절 각도 시각화

In [ ]:
GAIT_PHASES = [0, 25, 50, 75, 100]
PHASE_LABELS = ['IC\n(0%)', 'LR\n(25%)', 'MSt\n(50%)', 'TSt\n(75%)', 'PSw\n(100%)']

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# ── 상단: 시간 영역 (처음 2 사이클) ──────────────────────────
disp = knee_df[knee_df['time'] <= 2.0]
axes[0].plot(disp['time'], disp['knee_angle'], color=JOINT_COLORS['Knee'],
             linewidth=2, label='Measured Angle')
axes[0].axhline(knee_max, color='#E74C3C', linestyle='--', alpha=0.6,
                label=f'Max: {knee_max:.1f}°')
axes[0].axhline(knee_min, color='#27AE60', linestyle='--', alpha=0.6,
                label=f'Min: {knee_min:.1f}°')
axes[0].fill_between(disp['time'], knee_min, knee_max,
                     alpha=0.12, color='gold', label=f'ROM: {knee_rom:.1f}°')
axes[0].set(ylabel='Knee Angle (°)', title='무릎 관절 각도 — 시간 영역')
axes[0].legend(loc='upper right')

# ── 하단: 1 보행 사이클 ──────────────────────────────────────
cyc = knee_df[knee_df['time'] <= CYCLE_DUR]
axes[1].plot(cyc['gait_phase'], cyc['knee_angle'],
             color=JOINT_COLORS['Knee'], linewidth=2.5,
             marker='o', markersize=4, markevery=10)
axes[1].axhline(normal_rom, color='gray', linestyle=':', linewidth=2,
                label=f'Normal ROM: {normal_rom}°')
axes[1].set(xlabel='Gait Cycle (%)', ylabel='Knee Angle (°)',
            title='무릎 관절 각도 — 보행 사이클',
            xlim=(0, 100))
axes[1].set_xticks(GAIT_PHASES)
axes[1].set_xticklabels(PHASE_LABELS, fontsize=9)
axes[1].legend(loc='upper right')

fig.tight_layout()
save_fig(fig, '01_knee_angle_analysis.png')
plt.show()

save_csv(knee_df, '01_knee_angle_data.csv')
save_csv(rom_result, '01_rom_analysis.csv')


---
## 🟡 중급 (Intermediate)
### 2. 다중 관절 운동학 분석 (Hip · Knee · Ankle)


### 2-1. 다중 관절 데이터 생성 및 ROM 계산

In [ ]:
np.random.seed(42)
t, phase = make_time_gait()
phase_rad = phase / 100 * np.pi   # 편의용 라디안 변환 (0~π)

joint_signals = {
    'hip_angle':   10 + 20 * np.sin(phase_rad + np.pi/2) + 0.8 * np.random.randn(len(t)),
    'knee_angle':   5 + 15 * np.sin(phase_rad)           + 0.8 * np.random.randn(len(t)),
    'ankle_angle':      5 * np.sin(phase_rad - np.pi/4)  + 0.8 * np.random.randn(len(t)),
}

multi_df = pd.DataFrame({'time': t, 'gait_phase': phase, **joint_signals})

# ── 관절별 ROM 계산 ──────────────────────────────────────────
rom_rows = []
for name, col in zip(JOINT_NAMES, JOINT_COLS):
    hi, lo = multi_df[col].max(), multi_df[col].min()
    rom = hi - lo
    norm = NORMAL_ROMS[name]
    rom_rows.append({'Joint': name, 'Max (°)': hi, 'Min (°)': lo,
                     'ROM (°)': rom, 'Normal ROM (°)': norm,
                     'ROM % of Normal': rom / norm * 100})

rom_compare = pd.DataFrame(rom_rows)
print('관절별 ROM 비교:')
print(rom_compare.to_string(index=False))


### 2-2. 보행 이벤트별 관절 각도

In [ ]:
GAIT_EVENTS = {'Initial Contact':     0,
               'Loading Response':    12,
               'Mid Stance':          50,
               'Terminal Stance':     88,
               'Pre Swing':          100}

event_rows = []
for event, pct in GAIT_EVENTS.items():
    mask = np.abs(multi_df['gait_phase'] - pct) < 2
    seg  = multi_df[mask]
    if not seg.empty:
        event_rows.append({'Event': f'{event} ({pct}%)',
                           **{f'{n} (°)': seg[c].mean()
                              for n, c in zip(JOINT_NAMES, JOINT_COLS)}})

event_df = pd.DataFrame(event_rows)
print('보행 이벤트별 관절 각도:')
print(event_df.to_string(index=False))


### 2-3. 종합 운동학 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = [JOINT_COLORS[n] for n in JOINT_NAMES]

disp3 = multi_df[multi_df['time'] <= 3.0]
cyc1  = multi_df[multi_df['time'] <= CYCLE_DUR]

# ── (0,0) 시계열 ─────────────────────────────────────────────
for col, name, c in zip(JOINT_COLS, JOINT_NAMES, colors):
    axes[0, 0].plot(disp3['time'], disp3[col], color=c,
                    linewidth=1.5, alpha=0.85, label=name)
axes[0, 0].set(ylabel='Joint Angle (°)', title='3관절 시계열 (3 사이클)')
axes[0, 0].legend()

# ── (0,1) 보행 사이클 ────────────────────────────────────────
markers = ['o', 's', '^']
for col, name, c, mk in zip(JOINT_COLS, JOINT_NAMES, colors, markers):
    axes[0, 1].plot(cyc1['gait_phase'], cyc1[col], color=c,
                    linewidth=2, marker=mk, markersize=5,
                    markevery=10, label=name)
axes[0, 1].set(xlabel='Gait Cycle (%)', ylabel='Joint Angle (°)',
               title='보행 사이클별 관절 각도', xlim=(0, 100))
axes[0, 1].legend()

# ── (1,0) ROM 막대 ───────────────────────────────────────────
bars = axes[1, 0].bar(rom_compare['Joint'], rom_compare['ROM (°)'],
                      color=colors, alpha=0.75, edgecolor='k')
axes[1, 0].set(ylabel='ROM (°)', title='관절별 ROM')
for bar, row in zip(bars, rom_compare.itertuples()):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f"{row._4:.1f}°", ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── (1,1) ROM vs Normal ──────────────────────────────────────
x = np.arange(len(JOINT_NAMES))
w = 0.35
axes[1, 1].bar(x - w/2, rom_compare['ROM (°)'],        w, label='Measured', color='steelblue', alpha=0.8)
axes[1, 1].bar(x + w/2, rom_compare['Normal ROM (°)'], w, label='Normal',   color='#F39C12',  alpha=0.8)
axes[1, 1].set(ylabel='ROM (°)', title='측정 ROM vs 정상 ROM')
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(JOINT_NAMES)
axes[1, 1].legend()

fig.tight_layout()
save_fig(fig, '02_multi_joint_kinematics.png')
plt.show()

save_csv(multi_df,     '02_multi_joint_angles.csv')
save_csv(rom_compare,  '02_rom_comparison.csv')
save_csv(event_df,     '02_gait_events_angles.csv')


---
## 🔴 고급 (Advanced)
### 3. 치료 전후 비교 및 움직임 품질 평가


### 3-1. 치료 전후 운동학 데이터 생성

In [ ]:
# 치료 전: ROM 감소 + 노이즈 증가 / 치료 후: 정상에 근접
SESSION_PARAMS = {
    #         hip            knee         ankle   noise
    'pre':  dict(hip_amp=15, knee_amp=10, ank_amp=3, noise=2.0),
    'post': dict(hip_amp=20, knee_amp=15, ank_amp=5, noise=1.0),
}

def generate_patient_data(session: str) -> pd.DataFrame:
    """session = 'pre' | 'post' — 치료 전후 운동학 데이터 반환."""
    np.random.seed(42)
    t, phase = make_time_gait()
    rad = phase / 100 * np.pi
    p   = SESSION_PARAMS[session]
    n   = p['noise']

    return pd.DataFrame({
        'time':        t,
        'gait_phase':  phase,
        'hip_angle':   10 + p['hip_amp']  * np.sin(rad + np.pi/2) + n * np.random.randn(len(t)),
        'knee_angle':   5 + p['knee_amp'] * np.sin(rad)           + n * np.random.randn(len(t)),
        'ankle_angle':      p['ank_amp']  * np.sin(rad - np.pi/4) + n * np.random.randn(len(t)),
        'session':     session,
    })

pre_df  = generate_patient_data('pre')
post_df = generate_patient_data('post')

print('치료 전 데이터 샘플:')
print(pre_df.head(5).to_string(index=False))


### 3-2. 움직임 품질 점수 계산

In [ ]:
def movement_quality(df: pd.DataFrame) -> dict:
    """
    ROM 점수(60%) + 부드러움 점수(40%) → 종합 점수.
    부드러움: 잔차(측정값 - 부드러운 추정값)의 표준편차로 평가.
    """
    normal_roms = [NORMAL_ROMS[n] for n in JOINT_NAMES]

    rom_scores = []
    smooth_scores = []
    for col, n_rom in zip(JOINT_COLS, normal_roms):
        measured_rom = df[col].max() - df[col].min()
        rom_scores.append(min(measured_rom / n_rom * 100, 100))

        # 부드러움: 이동평균과의 잔차 RMS
        smooth_ref  = df[col].rolling(5, center=True, min_periods=1).mean()
        residual_std = (df[col] - smooth_ref).std()
        smooth_scores.append(max(0, 100 - residual_std * 15))

    rom_s    = float(np.mean(rom_scores))
    smooth_s = float(np.mean(smooth_scores))
    overall  = rom_s * 0.6 + smooth_s * 0.4
    return {'ROM Score': rom_s, 'Smoothness Score': smooth_s, 'Overall Score': overall}

pre_scores  = movement_quality(pre_df)
post_scores = movement_quality(post_df)

quality_df = pd.DataFrame([
    {'Session': 'Pre-Treatment',  **pre_scores},
    {'Session': 'Post-Treatment', **post_scores},
])
print('움직임 품질 점수:')
print(quality_df.to_string(index=False))

delta = post_scores['Overall Score'] - pre_scores['Overall Score']
print(f"\n전체 점수 개선: {delta:+.1f}점 ({delta / pre_scores['Overall Score'] * 100:+.1f}%)")


### 3-3. 통계 분석 (Paired t-test)

In [ ]:
# 관절별 ROM 추출
pre_roms  = [pre_df[c].max()  - pre_df[c].min()  for c in JOINT_COLS]
post_roms = [post_df[c].max() - post_df[c].min() for c in JOINT_COLS]

t_stat, p_val = stats.ttest_rel(post_roms, pre_roms)

stat_df = pd.DataFrame({
    'Joint':       JOINT_NAMES,
    'Pre ROM (°)': pre_roms,
    'Post ROM (°)': post_roms,
    'Change (°)':  [p - r for p, r in zip(post_roms, pre_roms)],
})
print('치료 전후 ROM 비교:')
print(stat_df.to_string(index=False))
print(f'\nPaired t-test — t = {t_stat:.3f},  p = {p_val:.4f}')
print('→ 통계적으로 유의미한 차이 있음 (p < 0.05)' if p_val < 0.05
      else '→ 통계적으로 유의미한 차이 없음 (p ≥ 0.05)')


### 3-4. 종합 치료 효과 대시보드

In [ ]:
pre_cyc  = pre_df[pre_df['time']  <= CYCLE_DUR]
post_cyc = post_df[post_df['time'] <= CYCLE_DUR]

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.32)

# ── 공통 플롯 함수 ───────────────────────────────────────────
def plot_joints(ax, df, title):
    markers = ['o', 's', '^']
    for col, name, c, mk in zip(JOINT_COLS, JOINT_NAMES,
                                [JOINT_COLORS[n] for n in JOINT_NAMES], markers):
        ax.plot(df['gait_phase'], df[col], color=c, linewidth=2,
                marker=mk, markersize=4, markevery=10, label=name)
    ax.set(xlabel='Gait Cycle (%)', ylabel='Angle (°)',
           title=title, xlim=(0, 100))
    ax.legend(fontsize=8)

# ── 행 0: 치료 전 / 후 / ROM 비교 ───────────────────────────
plot_joints(fig.add_subplot(gs[0, 0]), pre_cyc,  '치료 전 관절 각도')
plot_joints(fig.add_subplot(gs[0, 1]), post_cyc, '치료 후 관절 각도')

ax_rom = fig.add_subplot(gs[0, 2])
x = np.arange(len(JOINT_NAMES)); w = 0.35
ax_rom.bar(x - w/2, pre_roms,  w, label='Pre',  color='#E74C3C', alpha=0.75)
ax_rom.bar(x + w/2, post_roms, w, label='Post', color='#27AE60', alpha=0.75)
ax_rom.set(ylabel='ROM (°)', title='ROM 개선')
ax_rom.set_xticks(x); ax_rom.set_xticklabels(JOINT_NAMES, fontsize=9)
ax_rom.legend(fontsize=9)

# ── 행 1: 무릎 각도 비교 (전체 행) ──────────────────────────
ax_knee = fig.add_subplot(gs[1, :])
ax_knee.plot(pre_cyc['gait_phase'],  pre_cyc['knee_angle'],
             color='#E74C3C', linewidth=2.5, marker='o', markersize=4,
             markevery=10, label='Pre-Treatment')
ax_knee.plot(post_cyc['gait_phase'], post_cyc['knee_angle'],
             color='#27AE60', linewidth=2.5, marker='s', markersize=4,
             markevery=10, label='Post-Treatment')
ax_knee.fill_between(pre_cyc['gait_phase'],
                     pre_cyc['knee_angle'], post_cyc['knee_angle'],
                     alpha=0.15, color='gold')
ax_knee.set(xlabel='Gait Cycle (%)', ylabel='Knee Angle (°)',
            title='무릎 각도: 치료 전후 비교', xlim=(0, 100))
ax_knee.legend()

# ── 행 2: 품질 점수 / 세부 지표 / 요약 텍스트 ──────────────
ax_ov = fig.add_subplot(gs[2, 0])
sessions = ['Pre-Tx', 'Post-Tx']
scores   = [pre_scores['Overall Score'], post_scores['Overall Score']]
bars = ax_ov.bar(sessions, scores, color=['#E74C3C', '#27AE60'], alpha=0.8, edgecolor='k')
ax_ov.set(ylabel='Quality Score', title='전체 움직임 품질 점수', ylim=(0, 100))
for bar, s in zip(bars, scores):
    ax_ov.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               f'{s:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax_det = fig.add_subplot(gs[2, 1])
metrics = ['ROM\nScore', 'Smoothness\nScore']
ax_det.bar(np.arange(2) - w/2,
           [pre_scores['ROM Score'], pre_scores['Smoothness Score']],
           w, label='Pre-Tx',  color='#E74C3C', alpha=0.75)
ax_det.bar(np.arange(2) + w/2,
           [post_scores['ROM Score'], post_scores['Smoothness Score']],
           w, label='Post-Tx', color='#27AE60', alpha=0.75)
ax_det.set(ylabel='Score', title='세부 품질 지표', ylim=(0, 110))
ax_det.set_xticks(np.arange(2)); ax_det.set_xticklabels(metrics, fontsize=9)
ax_det.legend(fontsize=9)

ax_txt = fig.add_subplot(gs[2, 2])
ax_txt.axis('off')
summary_lines = [
    '── 치료 효과 요약 ──',
    f'전체 점수: {delta:+.1f}점',
    f'({delta / pre_scores["Overall Score"] * 100:+.1f}%)',
    '',
    'ROM 개선:',
    *[f'  {n}: {p - r:+.1f}°' for n, p, r in zip(JOINT_NAMES, post_roms, pre_roms)],
    '',
    f'Paired t-test:',
    f'  p = {p_val:.4f}',
    '  유의미 ✓' if p_val < 0.05 else '  비유의 ✗',
]
ax_txt.text(0.05, 0.95, '\n'.join(summary_lines),
            transform=ax_txt.transAxes, fontsize=10,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#FFF3CD', alpha=0.8))

save_fig(fig, '03_treatment_effect_dashboard.png')
plt.show()

# ── CSV 저장 ─────────────────────────────────────────────────
save_csv(pre_df,    '03_pre_treatment_data.csv')
save_csv(post_df,   '03_post_treatment_data.csv')
save_csv(quality_df,'03_quality_scores.csv')
save_csv(stat_df,   '03_statistical_comparison.csv')


---
## 📋 결론 및 임상 적용

### Week 6 학습 내용 요약

| 레벨 | 주요 내용 |
|------|----------|
| 🟢 초급 | 단일 관절(무릎) 각도 측정, ROM 계산, 보행 사이클 시각화 |
| 🟡 중급 | Hip·Knee·Ankle 동시 분석, 보행 이벤트별 각도, 운동학 프로파일 |
| 🔴 고급 | 치료 전후 비교, 움직임 품질 점수, Paired t-test, 임상 대시보드 |

### 임상 응용
- 관절 운동 범위 제한 진단 및 모니터링
- 재활 프로그램 효과 정량적 평가
- 치료 계획 수립 및 수정 근거 제공
- 환자 장기 추적 데이터베이스 구축


In [ ]:
summary_text = f"""
{'='*44}
         Week 6 분석 완료 요약
{'='*44}

저장 위치: {RUN_OUT}

생성 파일:
  초급 │ 01_knee_angle_data.csv
       │ 01_rom_analysis.csv
       │ 01_knee_angle_analysis.png
  중급 │ 02_multi_joint_angles.csv
       │ 02_rom_comparison.csv
       │ 02_gait_events_angles.csv
       │ 02_multi_joint_kinematics.png
  고급 │ 03_pre_treatment_data.csv
       │ 03_post_treatment_data.csv
       │ 03_quality_scores.csv
       │ 03_statistical_comparison.csv
       │ 03_treatment_effect_dashboard.png
{'='*44}
"""
print(summary_text)

(RUN_OUT / 'SUMMARY.txt').write_text(summary_text, encoding='utf-8')
print(f'요약 저장: {RUN_OUT}/SUMMARY.txt')
